# VisDrone - YOLO11m SAHI
1. Evalute YOLO11m base model on dataset VisDrone
2. Finetune and evaluate YOLO11m


## 1. Load Wheels & Create YAML 

In [1]:
import os
import sys
import subprocess
from pathlib import Path

# 1. Automatically scan the entire /kaggle directory to find the exact location of wheel files
WHEEL_DATASET_DIR = None
print("Searching for the wheels directory...")
for p in Path("/kaggle").rglob("ultralytics*.whl"):
    WHEEL_DATASET_DIR = p.parent
    break

if WHEEL_DATASET_DIR is None:
    raise FileNotFoundError(
        "❌ WHEELS DIRECTORY NOT FOUND!\n"
        "Possible reasons:\n"
        "1. You haven't added the wheels dataset (via 'Add Data' on the right panel).\n"
        "2. If you just downloaded the wheels, the /kaggle/working/wheels folder might have been cleared after a session reset."
    )
print(f"✅ Found wheels directory at: {WHEEL_DATASET_DIR}")

# Proceed with offline installation (using subprocess to ensure immediate module recognition)
print("Installing libraries...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-index", 
    f"--find-links={WHEEL_DATASET_DIR}", 
    "ultralytics", "pycocotools", "pandas", "matplotlib", "seaborn", "tqdm", "sahi"
], check=True)
print("✅ Library installation complete!")

# ==============================================================================
# 2. Import libraries after installation
# ==============================================================================
import json
import shutil
import time
import logging
import cv2
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

# --- Mute SAHI warnings regarding low confidence and NMS switching ---
logging.getLogger("sahi").setLevel(logging.ERROR)
print("✅ SAHI warnings muted.")

# ==============================================================================
# 3. GENERAL CONFIGURATION & DYNAMIC MODEL DISCOVERY
# ==============================================================================
SELECTED_MODEL = "yolo11m.pt"

# Find YOLO weights file
BASE_MODEL_PATH = str(Path("/kaggle/working") / SELECTED_MODEL)
if not Path(BASE_MODEL_PATH).exists():
    # Scan for the model throughout /kaggle
    model_candidates = list(Path("/kaggle").rglob(SELECTED_MODEL))
    if model_candidates:
        BASE_MODEL_PATH = str(model_candidates[0])
    else:
        raise FileNotFoundError(f"❌ Weights file not found: {SELECTED_MODEL}. Please check your dataset!")

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}, Model path: {BASE_MODEL_PATH}")

# ==============================================================================
# 4. LOCATE VISDRONE DATASET AND CREATE YAML
# ==============================================================================
print("Searching for the VisDrone dataset directory...")
DATA_ROOT = None
# Scan for the directory containing standard VisDrone structure
for p in Path('/kaggle/input').rglob('*'):
    if p.is_dir() and (p / 'images/train').exists() and (p / 'images/val').exists():
        DATA_ROOT = p
        break

if not DATA_ROOT:
    raise FileNotFoundError("❌ VisDrone dataset (must contain 'images/train') not found. Did you Add Data?")

print(f"✅ Found VisDrone dataset at: {DATA_ROOT}")

# Required paths
TEST_IMAGE_DIR = DATA_ROOT / "images" / "test-dev"
VAL_IMAGE_DIR  = DATA_ROOT / "images" / "val"
WORK_ROOT = Path("/kaggle/working/visdrone_experiments")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

# Automatically generate YOLO YAML configuration file
yaml_content = f"""
path: {DATA_ROOT.as_posix()}
train: images/train
val: images/val
test: images/test-dev

nc: 10
names: ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor']
"""

yaml_path = Path('/kaggle/working/visdrone_corrected.yaml')
yaml_path.write_text(yaml_content)
print(f"✅ YOLO YAML configuration automatically created at: {yaml_path}")

Searching for the wheels directory...
✅ Found wheels directory at: /kaggle/input/datasets/tranphungdinh/visdrone-yolo11n-wheels/visdrone_yolo_wheels
Installing libraries...
✅ Library installation complete!
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ SAHI warnings muted.
✅ Using device: cuda:0, Model path: /kaggle/input/datasets/tranphungdinh/visdrone-yolo11n-wheels/yolo11m.pt
Searching for the VisDrone dataset directory...
✅ Found VisDrone dataset at: /kaggle/input/datasets/tranphungdinh/visdrone-yolo-format/VisDrone
✅ YOLO YAML configuration automatically created at: /kaggle/working/visdrone_corrected.yaml


In [2]:
# ============================================================
# FUNCTION: EVALUATE & SAVE METRICS IN COCO FORMAT (WITH SAHI)
# ============================================================
def evaluate_and_save(model, val_data_yaml, task_name, output_dir, val_images_dir, use_sahi=True, conf=0.01):
    """
    Evaluates the model. If use_sahi=True, it replaces the standard 1280 inference 
    with SAHI 640x640 slicing. 
    NOTE: Confidence is raised to 0.01 to avoid the 0.001 threshold warning, 
    but kept low enough to ensure valid mAP curves.
    """
    print(f"\n==================================================")
    print(f"STARTING EVALUATION: {task_name}")
    print(f"==================================================")
    
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # 1. Run Ultralytics Validation to generate Ground Truth JSON natively
    print("Running YOLO validation to generate ground truth and base predictions...")
    results = model.val(
        data=val_data_yaml,
        imgsz=1280 if not use_sahi else 640, # Shrink base eval if SAHI overrides it anyway
        max_det=500,
        save_json=True,
        project=output_dir,
        name='val_run'
    )
    
    run_dir = Path(results.save_dir)
    pred_json = run_dir / "predictions.json"
    gt_json = run_dir / "val_annotations.json"
    
    # 2. Overwrite predictions.json using SAHI Sliced Inference
    if use_sahi and gt_json.exists():
        print(f"Applying SAHI slicing (640x640) for inference (Conf: {conf})...")
        
        # Load GT to match image IDs accurately for PyCOCOtools
        with open(gt_json, 'r') as f:
            gt_data = json.load(f)
            
        # Initialize SAHI Model
        # Setting postprocess explicitly to NMS/IOU prevents internal SAHI fallback warnings
        detection_model = AutoDetectionModel.from_pretrained(
            model_type='yolov8',
            model_path=model.pt_path if hasattr(model, 'pt_path') else model.ckpt_path,
            confidence_threshold=conf,
            device="cuda:0" if torch.cuda.is_available() else "cpu"
        )
        
        sahi_predictions = []
        for img_info in tqdm(gt_data['images'], desc="SAHI Inference"):
            img_path = Path(val_images_dir) / img_info['file_name']
            
            # Perform Sliced Prediction
            result = get_sliced_prediction(
                str(img_path),
                detection_model,
                slice_height=640,
                slice_width=640,
                overlap_height_ratio=0.2,
                overlap_width_ratio=0.2,
                postprocess_type="NMS",
                postprocess_match_metric="IOU"
            )
            
            # Convert results to COCO JSON format
            for obj in result.object_prediction_list:
                x = obj.bbox.minx
                y = obj.bbox.miny
                w = obj.bbox.maxx - x
                h = obj.bbox.maxy - y
                sahi_predictions.append({
                    "image_id": img_info['id'],
                    "category_id": obj.category.id,
                    "bbox": [x, y, w, h],
                    "score": obj.score.value
                })
                
        # Overwrite YOLO's predictions.json with SAHI's predictions
        with open(pred_json, 'w') as f:
            json.dump(sahi_predictions, f)
        print("✅ SAHI inference completed and saved to predictions.json")

    # Copy prediction.json to output directory for easy access
    saved_pred_path = Path(output_dir) / f"{task_name}_predictions.json"
    if pred_json.exists():
        shutil.copy2(pred_json, saved_pred_path)
        
    # 3. Save Per-Class Metrics (Reflects initial YOLO validation run)
    class_names = results.names
    per_class_df = pd.DataFrame({
        'Class_ID': results.box.ap_class_index,
        'Class_Name': [class_names[i] for i in results.box.ap_class_index],
        'mAP_50': results.box.ap50,  
        'mAP_50_95': results.box.ap  
    })
    per_class_path = Path(output_dir) / f"{task_name}_per_class_metrics.csv"
    per_class_df.to_csv(per_class_path, index=False)
    
    # 4. Extract Overall COCO Metrics using PyCOCOtools
    print("Calculating COCO Metrics (Evaluating SAHI Predictions)...")
    if gt_json.exists() and pred_json.exists():
        cocoGt = COCO(str(gt_json))
        cocoDt = cocoGt.loadRes(str(pred_json))
        cocoEval = COCOeval(cocoGt, cocoDt, 'bbox')
        cocoEval.params.maxDets = [1, 10, 100, 500] # Set max detections for AR calculation
        cocoEval.evaluate()
        cocoEval.accumulate()
        cocoEval.summarize()
        
        stats = cocoEval.stats
        metrics_dict = {
            "AP (50-95)": float(stats[0]), "AP50": float(stats[1]), "AP75": float(stats[2]),
            "AP small": float(stats[3]), "AP medium": float(stats[4]), "AP large": float(stats[5]),
            "AR@1": float(stats[6]), "AR@10": float(stats[7]), "AR@100": float(stats[8]),
            "AR@500": float(stats[9]), "AR small": float(stats[10]), "AR medium": float(stats[11]),
            "AR large": float(stats[12])
        }
        
        metrics_path = Path(output_dir) / f"{task_name}_overall_metrics.json"
        with open(metrics_path, 'w') as f:
            json.dump(metrics_dict, f, indent=4)
        print(f"✅ Saved overall metrics at: {metrics_path}")
    else:
        print("⚠️ Could not find JSON files for PyCOCOtools calculation.")

# ============================================================
# OPTIONAL: PURE SAHI PREDICTION FUNCTION FOR NEW IMAGES
# ============================================================
def predict_sahi_real_world(model_path, image_path, output_dir, conf=0.25):
    """
    Use this function for pure inference/prediction on new images.
    Confidence is set higher (0.25) to avoid drawing noisy bounding boxes.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    detection_model = AutoDetectionModel.from_pretrained(
        model_type='yolov8',
        model_path=model_path,
        confidence_threshold=conf,
        device="cuda:0" if torch.cuda.is_available() else "cpu"
    )
    result = get_sliced_prediction(
        image_path,
        detection_model,
        slice_height=640, slice_width=640,
        overlap_height_ratio=0.2, overlap_width_ratio=0.2,
        postprocess_type="NMS",
        postprocess_match_metric="IOU"
    )
    out_file = Path(output_dir) / f"sahi_pred_{Path(image_path).name}"
    result.export_visuals(export_dir=str(output_dir), file_name=out_file.stem)
    print(f"✅ Saved SAHI prediction visual to: {out_file}")

## 2. Evaluate base YOLO11m SAHI

In [3]:
# ============================================================
# TASK 1: EVALUATE BASE MODEL YOLO11m (WITH SAHI)
# ============================================================
# Initialize base model from offline downloaded weights
base_model = YOLO(BASE_MODEL_PATH)
# Store pt_path dynamically to access it inside evaluate_and_save
base_model.pt_path = BASE_MODEL_PATH

evaluate_and_save(
    model=base_model,
    val_data_yaml=str(yaml_path),
    task_name="Base_yolo11m_SAHI",
    output_dir="/kaggle/working/eval_base_sahi",
    val_images_dir=str(VAL_IMAGE_DIR),
    use_sahi=True,
    conf=0.01 # Increased slightly from 0.001 to disable warnings, maintains precise mAP
)


STARTING EVALUATION: Base_yolo11m_SAHI
Running YOLO validation to generate ground truth and base predictions...
Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
YOLO11m summary (fused): 125 layers, 20,091,712 parameters, 0 gradients, 68.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 17.3±5.4 MB/s, size: 117.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /kaggle/input/datasets/tranphungdinh/visdrone-yolo-format/VisDrone/labels/val... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 521.8it/s 1.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/tranphungdinh/visdrone-yolo-format/VisDrone/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 10.5it/s 3.3s
   

## 3. Finetune YOLO 11m

In [4]:
# ============================================================
# TASK 4.1: SLICE DATASET FOR SAHI FINETUNING
# ============================================================
import os
import cv2
from pathlib import Path
from tqdm.auto import tqdm

def slice_yolo_dataset(img_dir, lbl_dir, out_img_dir, out_lbl_dir, slice_size=640, overlap=0.2):
    """
    Slices images and corresponding YOLO bounding boxes into smaller patches.
    """
    Path(out_img_dir).mkdir(parents=True, exist_ok=True)
    Path(out_lbl_dir).mkdir(parents=True, exist_ok=True)
    
    step = int(slice_size * (1 - overlap))
    img_paths = list(Path(img_dir).glob("*.jpg"))
    
    for img_path in tqdm(img_paths, desc=f"Slicing {Path(img_dir).parent.name}"):
        lbl_path = Path(lbl_dir) / f"{img_path.stem}.txt"
        
        img = cv2.imread(str(img_path))
        if img is None: continue
        h, w = img.shape[:2]
        
        # 1. Read existing bounding boxes
        boxes = []
        if lbl_path.exists():
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        c, x, y, bw, bh = map(float, parts)
                        # Convert YOLO relative format to absolute [x1, y1, x2, y2]
                        x1, y1 = (x - bw/2) * w, (y - bh/2) * h
                        x2, y2 = (x + bw/2) * w, (y + bh/2) * h
                        boxes.append([int(c), x1, y1, x2, y2])
        
        # 2. Slice the image
        for y_start in range(0, h, step):
            for x_start in range(0, w, step):
                y_end = min(y_start + slice_size, h)
                x_end = min(x_start + slice_size, w)
                
                # Adjust to ensure exact slice_size if hitting the edge
                if y_end - y_start < slice_size and h >= slice_size: y_start = h - slice_size
                if x_end - x_start < slice_size and w >= slice_size: x_start = w - slice_size
                    
                slice_img = img[int(y_start):int(y_end), int(x_start):int(x_end)]
                slice_boxes = []
                sw, sh = x_end - x_start, y_end - y_start
                
                # 3. Process boxes for this specific slice
                for box in boxes:
                    c, bx1, by1, bx2, by2 = box
                    # Intersect original box with the current slice
                    ix1, iy1 = max(x_start, bx1), max(y_start, by1)
                    ix2, iy2 = min(x_end, bx2), min(y_end, by2)
                    
                    # Keep if the box is valid and not a 1-pixel sliver
                    if ix1 < ix2 and iy1 < iy2 and (ix2 - ix1) > 3 and (iy2 - iy1) > 3:
                        # Convert back to YOLO format (relative to slice)
                        rx, ry = ((ix1 + ix2) / 2 - x_start) / sw, ((iy1 + iy2) / 2 - y_start) / sh
                        rbw, rbh = (ix2 - ix1) / sw, (iy2 - iy1) / sh
                        slice_boxes.append(f"{c} {rx:.6f} {ry:.6f} {rbw:.6f} {rbh:.6f}")
                
                # 4. Save slice only if it contains objects (prevents heavy background imbalance)
                if len(slice_boxes) > 0:
                    slice_name = f"{img_path.stem}_{x_start}_{y_start}"
                    cv2.imwrite(str(Path(out_img_dir) / f"{slice_name}.jpg"), slice_img)
                    with open(Path(out_lbl_dir) / f"{slice_name}.txt", 'w') as f:
                        f.write('\n'.join(slice_boxes))

# Execute Slicing
SLICED_ROOT = Path("/kaggle/working/visdrone_experiments/VisDrone_Sliced")
slice_yolo_dataset(
    DATA_ROOT / "images/train", DATA_ROOT / "labels/train", 
    SLICED_ROOT / "train/images", SLICED_ROOT / "train/labels", slice_size=640
)
slice_yolo_dataset(
    DATA_ROOT / "images/val", DATA_ROOT / "labels/val", 
    SLICED_ROOT / "val/images", SLICED_ROOT / "val/labels", slice_size=640
)

# Generate new YAML for the sliced dataset
sliced_yaml_content = f"""
path: {SLICED_ROOT.as_posix()}
train: train/images
val: val/images
test: val/images

nc: 10
names: ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor']
"""
sliced_yaml_path = Path('/kaggle/working/visdrone_sliced.yaml')
sliced_yaml_path.write_text(sliced_yaml_content)
print(f"✅ Sliced dataset prepared. YAML created at: {sliced_yaml_path}")

Slicing images:   0%|          | 0/6471 [00:00<?, ?it/s]

Slicing images:   0%|          | 0/548 [00:00<?, ?it/s]

✅ Sliced dataset prepared. YAML created at: /kaggle/working/visdrone_sliced.yaml


In [5]:
# ============================================================
# TASK 4.2: TRAIN ON SLICED DATASET (640x640)
# ============================================================
print("\n==================================================")
print("STARTING YOLO11m FINETUNING ON SLICED SAHI DATA")
print("==================================================")

# Initialize model from base weights
sahi_train_model = YOLO(BASE_MODEL_PATH)

# Train using the new sliced YAML and smaller image size
sahi_train_results = sahi_train_model.train(
    data=str(sliced_yaml_path),
    epochs=15,             # Adjustable
    resume=False,          
    imgsz=640,             # Smaller size because images are already sliced
    batch=16,              # Increased batch size to match smaller imgsz
    max_det=500,
    cache=True,
    project='/kaggle/working/visdrone_experiments',
    name='finetune_sahi_640',
    save=True
)

run_dir_sahi = Path(sahi_train_results.save_dir)
print(f"✅ Best sliced model weights saved at: {run_dir_sahi / 'weights' / 'best.pt'}")


STARTING YOLO11m FINETUNING ON SLICED SAHI DATA
Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/visdrone_sliced.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=500, mixup=0.0, mode=train, model=/ka

## 4. Evaluate Fineuned Model

In [6]:
# --- Evaluate the new finetuned SAHI model ---
best_sahi_weight = run_dir_sahi / "weights" / "best.pt"
finetuned_sahi_model = YOLO(str(best_sahi_weight))
finetuned_sahi_model.pt_path = str(best_sahi_weight)

evaluate_and_save(
    model=finetuned_sahi_model,
    val_data_yaml=str(yaml_path), # Evaluate on ORIGINAL unsliced validation set
    task_name="Finetune_SAHI_640",
    output_dir="/kaggle/working/eval_finetune_sahi_640",
    val_images_dir=str(VAL_IMAGE_DIR),
    use_sahi=True,
    conf=0.01
)

print("\n✅ SAHI Finetuning and Evaluation Complete!")


STARTING EVALUATION: Finetune_SAHI_640
Running YOLO validation to generate ground truth and base predictions...
Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
YOLO11m summary (fused): 126 layers, 20,037,742 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.8 ms, read: 168.9±55.5 MB/s, size: 182.6 KB)
val: Scanning /kaggle/input/datasets/tranphungdinh/visdrone-yolo-format/VisDrone/labels/val... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548 2.0Kit/s 0.3s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/tranphungdinh/visdrone-yolo-format/VisDrone/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 12.3it/s 2.9s
                   all        548      38759      0.565      0.408      0.417      0.248
            pedestrian        520       8844      0.615      0.405   